In [56]:
import json
import boto3
import os
import re
from dotenv import load_dotenv

load_dotenv()

class AssignmentEditor:
    def __init__(self):
        self.s3 = boto3.client(
            "s3",
            aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
            aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
            region_name="ap-south-1"
        )

        self.dynamodb = boto3.resource(
            "dynamodb",
            aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
            aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
            region_name="ap-south-1"
        )

        self.BUCKET = "generated-assignments-1"

        # 🔥 THIS IS THE TABLE IT CHECKS
        self.TABLE = self.dynamodb.Table("Assignments")

        self.EDITOR_INDEX_KEY = "editor/custom-editor/index.html"
        self.EDITOR_BASE_URL = (
            "https://generated-assignments.s3.amazonaws.com/editor/custom-editor/"
        )

    # -------------------------
    # Fetch assignment from DynamoDB
    # -------------------------
    def get_assignment(self, assignment_id):
        res = self.TABLE.get_item(
            Key={"assignment_id": assignment_id}
        )
        if "Item" not in res:
            raise Exception("Assignment not found in assignments table")
        return res["Item"]

    # -------------------------
    # Read HTML from S3
    # -------------------------
    def read_html(self, key):
        obj = self.s3.get_object(Bucket=self.BUCKET, Key=key)
        return obj["Body"].read().decode("utf-8")

    # -------------------------
    # Extract BODY only
    # -------------------------
    def extract_body(self, html):
        match = re.search(
            r"<body[^>]*>(.*?)</body>",
            html,
            re.IGNORECASE | re.DOTALL
        )
        return match.group(1).strip() if match else html

    # -------------------------
    # Integrate editor + content
    # -------------------------
    def integrate_editor(self, editor_html, content_html):
        # Fix asset paths
        editor_html = editor_html.replace(
            "<head>",
            f'<head><base href="{self.EDITOR_BASE_URL}">'
        )

        safe_content = self.extract_body(content_html)

        marker = '<div id="editor"></div>'
        if marker not in editor_html:
            raise Exception('<div id="editor"></div> missing in editor index.html')

        return editor_html.replace(
            marker,
            f'<div id="editor" contenteditable="true">{safe_content}</div>'
        )

    # -------------------------
    # OPEN EDITOR API
    # -------------------------
    def open_editor(self, event):
        try:
            if "request" not in event or "body" not in event["request"]:
                raise Exception("Invalid request structure")

            body = json.loads(event["request"]["body"])

            assignment_id = body.get("assignment_id")
            if not assignment_id:
                raise Exception("assignment_id is required")

            # 1️⃣ Get assignment from DB
            assignment = self.get_assignment(assignment_id)

            # 2️⃣ Canonical key ONLY from DB
            s3_key = assignment.get("canonical_s3_key")
            if not s3_key:
                raise Exception("canonical_s3_key missing in assignment record")

            # 3️⃣ Load files
            content_html = self.read_html(s3_key)
            editor_html = self.read_html(self.EDITOR_INDEX_KEY)

            # 4️⃣ Merge editor + content
            final_html = self.integrate_editor(editor_html, content_html)

            return {
                "statusCode": 200,
                "headers": {
                    "Content-Type": "text/html; charset=utf-8",
                    "Access-Control-Allow-Origin": "*"
                },
                "body": final_html,
                "isBase64Encoded": False
            }

        except Exception as e:
            return {
                "statusCode": 400,  # 👈 NOT 500 (client error)
                "headers": {
                    "Access-Control-Allow-Origin": "*"
                },
                "body": json.dumps({"error": str(e)}),
                "isBase64Encoded": False
            }



def lambda_handler(event, context=None):
    return AssignmentEditor().open_editor(event)


In [57]:
testevent = {
  "request": {
    "body": json.dumps({
      "assignment_id": "a-123"
    })
  }
}


In [58]:
{
  "assignment_id": "a-123",
  "canonical_s3_key": "AB1C123/cse/teacher1@college.edu/20251105054612_51b830d88a/content.html"
}


{'assignment_id': 'a-123',
 'canonical_s3_key': 'AB1C123/cse/teacher1@college.edu/20251105054612_51b830d88a/content.html'}

In [59]:
lambda_handler(testevent)

{'statusCode': 400,
 'headers': {'Access-Control-Allow-Origin': '*'},
 'body': '{"error": "Assignment not found in assignments table"}',
 'isBase64Encoded': False}